<a href="https://colab.research.google.com/github/jatinjadon/heterogenous-kg-recommender/blob/main/GATv2_recommendation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import torch
import os
import urllib.request
import zipfile
import random
import numpy as np

In [20]:
def seed_everything(seed=42):
    # Lock standard Python random operations
    random.seed(seed)

    # Lock OS environment randomness
    os.environ['PYTHONHASHSEED'] = str(seed)

    # Lock Pandas/NumPy randomness
    np.random.seed(seed)

    # Lock PyTorch CPU randomness
    torch.manual_seed(seed)

    # Lock PyTorch GPU randomness
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # Force PyTorch to use deterministic algorithms (Fixes scatter_add)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(42)

In [21]:
# Download the dataset automatically if it doesn't exist
if not os.path.exists('ml-latest-small.zip'):
    url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
    urllib.request.urlretrieve(url, 'ml-latest-small.zip')

if not os.path.exists('ml-latest-small/ratings.csv'):
    with zipfile.ZipFile('ml-latest-small.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

# FETCH TMDB - Updated with raw URL to fix BadZipFile error
tmdb_zip_url = 'https://github.com/jatinjadon/heterogenous-kg-recommender/raw/main/tmdb.zip'

# If a previous failed download exists (the HTML file), remove it first
if os.path.exists('tmdb.zip') and not zipfile.is_zipfile('tmdb.zip'):
    os.remove('tmdb.zip')

if not os.path.exists('tmdb.zip'):
    urllib.request.urlretrieve(tmdb_zip_url, 'tmdb.zip')

if not os.path.exists('tmdb_5000_movies.csv'):
    with zipfile.ZipFile('tmdb.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
print("Datasets ready.")

Datasets ready.


In [22]:
!pip install torch_geometric

In [23]:
df_movies = pd.read_csv("tmdb_5000_movies.csv")
df_credits = pd.read_csv("tmdb_5000_credits.csv")
df_ratings = pd.read_csv('ml-latest-small/ratings.csv')
df_links = pd.read_csv('ml-latest-small/links.csv')
df_movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [24]:
import ast

print("Untangling JSON columns...")

# Convert the giant text strings into real Python lists of dictionaries
df_credits['cast'] = df_credits['cast'].apply(ast.literal_eval)
df_credits['crew'] = df_credits['crew'].apply(ast.literal_eval)

# Extract Actors (From the 'cast' column)
# We loop through the list of dictionaries and grab the 'name' value.
df_credits['actors'] = df_credits['cast'].apply(
    lambda cast_list: [actor['name'] for actor in cast_list[:5]]
)

# Extract Directors (From the 'crew' column)
# We add an IF statement to only grab the names where the job is 'Director'.
df_credits['directors'] = df_credits['crew'].apply(
    lambda crew_list: [member['name'] for member in crew_list if member['job'] == 'Director']
)
df_movies['genres'] = df_movies['genres'].apply(ast.literal_eval)
df_movies['genres'] = df_movies['genres'].apply(lambda genre_list: [genre['name'] for genre in genre_list])

df_movies['id'] = df_movies['id'].astype(int)

# Explode and extract the unique entities just like before
# Explode means it will add all the names in single np array of all columns
unique_actors = df_credits['actors'].explode().dropna().unique()
unique_directors = df_credits['directors'].explode().dropna().unique()
unique_movies_ids = df_movies['id'].unique()
unique_genres = df_movies['genres'].explode().unique()

num_actors = len(unique_actors)
num_directors = len(unique_directors)
num_movies_ids = len(unique_movies_ids)
num_genres = len(unique_genres)

print(f"Successfully extracted {num_movies_ids} unique movies!")
print(f"Successfully extracted {num_directors} unique directors!")
print(f"Successfully extracted {num_actors} unique actors!")
print(f"Successfully extracted {num_genres} unique genres!")

Untangling JSON columns...
Successfully extracted 4803 unique movies!
Successfully extracted 2577 unique directors!
Successfully extracted 9390 unique actors!
Successfully extracted 21 unique genres!


In [25]:
# Rename 'movie_id' in the credits dataframe to 'id' so they match perfectly
df_credits = df_credits.rename(columns={'movie_id': 'id'})

# Merge them together into one master table based on that unique 'id'
df_master = df_movies.merge(df_credits, on='id')

In [26]:
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

data = HeteroData()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

movie_mapping = {old_id: i for i, old_id in enumerate(unique_movies_ids)}
all_people = set(unique_actors).union(set(unique_directors))
person_mapping = {name: i for i, name in enumerate(all_people)}
genre_mapping = {name: i for i, name in enumerate(unique_genres)}

data['movie'].num_nodes = len(movie_mapping)
data['person'].num_nodes = len(person_mapping)
data['genre'].num_nodes = len(genre_mapping)

acted_in_src, acted_in_dst = [], []
directed_src, directed_dst = [], []
has_genre_src, has_genre_dst = [], []

for _, row in df_master.iterrows():
    m_idx = movie_mapping[row['id']]
    for actor in row['actors']:
        acted_in_src.append(person_mapping[actor])
        acted_in_dst.append(m_idx)
    for director in row['directors']:
        directed_src.append(person_mapping[director])
        directed_dst.append(m_idx)
    for genre in row['genres']:
        has_genre_src.append(m_idx)
        has_genre_dst.append(genre_mapping[genre])

data['person', 'acted_in', 'movie'].edge_index = torch.tensor([acted_in_src, acted_in_dst], dtype=torch.long)
data['person', 'directed', 'movie'].edge_index = torch.tensor([directed_src, directed_dst], dtype=torch.long)
data['movie', 'has_genre', 'genre'].edge_index = torch.tensor([has_genre_src, has_genre_dst], dtype=torch.long)

print("Core Knowledge Graph initialized.")

Core Knowledge Graph initialized.


In [27]:
import torch.nn as nn

In [28]:
df_user_actions = df_ratings.merge(df_links, on='movieId')
valid_actions = df_user_actions[df_user_actions['tmdbId'].isin(movie_mapping.keys())].copy() # only those users who have rated movies that r present in tmdb dataset

# Filter for active users
user_counts = valid_actions['userId'].value_counts() # user_counts denotes which user has rated how many movies
valid_users = user_counts[user_counts >= 5].index # which have rated more than 5 movies to reduce data sparsity
valid_actions = valid_actions[valid_actions['userId'].isin(valid_users)]

user_mapping = {old_id: i for i, old_id in enumerate(valid_actions['userId'].unique())}
data['user'].num_nodes = len(user_mapping)

u_src = [user_mapping[r['userId']] for _, r in valid_actions.iterrows()]
m_dst = [movie_mapping[r['tmdbId']] for _, r in valid_actions.iterrows()]

data['user', 'liked', 'movie'].edge_index = torch.tensor([u_src, m_dst], dtype=torch.long)

# Convert to bidirectional graph and move to device
data = T.ToUndirected()(data)
data = data.to(device)

print(f"Final Graph Structure:\n{data}")

Final Graph Structure:
HeteroData(
  movie={ num_nodes=4803 },
  person={ num_nodes=11705 },
  genre={ num_nodes=21 },
  user={ num_nodes=610 },
  (person, acted_in, movie)={ edge_index=[2, 23594] },
  (person, directed, movie)={ edge_index=[2, 5166] },
  (movie, has_genre, genre)={ edge_index=[2, 12160] },
  (user, liked, movie)={ edge_index=[2, 70194] },
  (movie, rev_acted_in, person)={ edge_index=[2, 23594] },
  (movie, rev_directed, person)={ edge_index=[2, 5166] },
  (genre, rev_has_genre, movie)={ edge_index=[2, 12160] },
  (movie, rev_liked, user)={ edge_index=[2, 70194] }
)


In [29]:
# Define the transform to split the data
# We only target the 'rates' edge for masking.
# We enable negative sampling so the model learns to identify
# not just what a user likes, but what they likely DON'T like.
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.2,
    edge_types=[('user', 'liked', 'movie')],
    rev_edge_types=[('movie', 'rev_liked', 'user')],
    add_negative_train_samples=True, # Critical for recommendation performance
    neg_sampling_ratio=1.0           # 1 negative sample per 1 positive edge
)

# Execute the split
train_data, val_data, test_data = transform(data)

# Verification
print(f"Training Interaction Edges:   {train_data['user', 'liked', 'movie'].edge_label_index.size(1)}")
print(f"Validation Interaction Edges: {val_data['user', 'liked', 'movie'].edge_label_index.size(1)}")
print(f"Testing Interaction Edges:    {test_data['user', 'liked', 'movie'].edge_label_index.size(1)}")

Training Interaction Edges:   98274
Validation Interaction Edges: 14038
Testing Interaction Edges:    28076


In [30]:
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, to_hetero

class GATv2Encoder(nn.Module):
    def __init__(self, hidden_channels, out_channels, num_heads=4):
        super().__init__()

        # LAYER 1: MULTI-HEAD EXPLORATION
        # Output dimension becomes: hidden_channels * num_heads (e.g., 64 * 4 = 256)
        self.conv1 = GATv2Conv(
            (-1, -1),
            hidden_channels,
            heads=num_heads,
            concat=True,         # Explicitly glue the 4 heads together
            add_self_loops=False
        )

        # LAYER 2: SINGLE-HEAD SYNTHESIS
        # Takes the 256-D input and compresses it down to the final 64-D output
        self.conv2 = GATv2Conv(
            (-1, -1),
            out_channels,
            heads=1,             # 1 Head for final compression
            concat=False,        # No concatenation needed for 1 head
            add_self_loops=False
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [31]:
class EdgeDecoder(nn.Module):
    def forward(self, z_dict, edge_label_index):
        # z_dict is a dictionary containing the final embeddings for all node types
        # Look at the "quiz" tensor to see which specific users and movies we are testing
        user_node_indices = edge_label_index[0]
        movie_node_indices = edge_label_index[1]

        # Extract those exact embeddings from the dictionary
        user_embeddings = z_dict['user'][user_node_indices]
        movie_embeddings = z_dict['movie'][movie_node_indices]

        # Calculate the Dot Product
        # A high positive number = strong link. A negative/low number = weak/no link.
        return (user_embeddings * movie_embeddings).sum(dim=-1)

In [32]:
class RecommendationModel(nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata):
        super().__init__()

        self.node_embeddings = nn.ModuleDict({
            'movie': nn.Embedding(data['movie'].num_nodes, hidden_channels),
            'person': nn.Embedding(data['person'].num_nodes, hidden_channels),
            'genre': nn.Embedding(data['genre'].num_nodes, hidden_channels),
            'user': nn.Embedding(data['user'].num_nodes, hidden_channels)
        })
        # Instantiate the base encoder and convert it to handle Heterogeneous data
        # metadata denotes knowledge graph having 4 different edge-types, it's required for to_hetero
        # to_hetero converts single encoder into different encoders for each edge type, as different weight matrices are required for each edge type
        base_encoder = GATv2Encoder(hidden_channels, out_channels)
        self.encoder = to_hetero(base_encoder, metadata, aggr='sum')
        self.decoder = EdgeDecoder()

    def forward(self, edge_index_dict, edge_label_index):
        x_dict = {
            'movie': self.node_embeddings['movie'].weight,
            'person': self.node_embeddings['person'].weight,
            'genre': self.node_embeddings['genre'].weight,
            'user': self.node_embeddings['user'].weight
        }
        # The Encoder uses the structure to learn the DNA
        # z_dict is a dictionary containing the final embeddings for all node types
        z_dict = self.encoder(x_dict, edge_index_dict)

        # The Decoder takes the DNA and answers the specific link prediction questions
        return self.decoder(z_dict, edge_label_index)

# We pass data.metadata() so PyG knows exactly how many edge types you have
model = RecommendationModel(hidden_channels=128, out_channels=64, metadata=data.metadata())
model = model.to(device)

In [33]:
from sklearn.metrics import roc_auc_score

# Initialize Optimizer and Loss Function
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()

# Define the Evaluation Function
def test(data_split):
    model.eval() # Put model in evaluation mode (turns off dropout, etc.)
    with torch.no_grad(): # Turn off gradient tracking to save memory

        # Pass the data through the model
        out = model(
            data_split.edge_index_dict,
            data_split['user', 'liked', 'movie'].edge_label_index
        )

        # Get the ground truth labels
        target = data_split['user', 'liked', 'movie'].edge_label

        # Calculate ROC-AUC score
        # We apply sigmoid to the outputs to convert them to probabilities for sklearn
        preds = torch.sigmoid(out).cpu().numpy()
        targets = target.cpu().numpy()

        return roc_auc_score(targets, preds)

print("--- Starting Training ---")

# The Main Training Loop
epochs = 100
for epoch in range(1, epochs + 1):
    model.train()           # Put model in training mode
    optimizer.zero_grad()   # Clear old gradients from the previous step

    # Forward Pass: Ask the model to predict the training edges
    out = model(
        train_data.edge_index_dict,
        train_data['user', 'liked', 'movie'].edge_label_index
    )

    # Get the ground truth training labels (the 1s and 0s)
    target = train_data['user', 'liked', 'movie'].edge_label

    # Calculate how wrong the predictions were
    loss = criterion(out, target)

    # Backward Pass: Calculate the gradients (the direction to adjust the weights)
    loss.backward()

    # Optimizer Step: Actually update the neural network's weights
    optimizer.step()

    # Every 10 epochs, test the model to see how it's doing
    if epoch % 10 == 0:
        val_auc = test(val_data)
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}')

# The Final Exam
test_auc = test(test_data)
print(f'\n--- Training Complete ---')
print(f'Final Test AUC: {test_auc:.4f}')

--- Starting Training ---
Epoch: 010, Loss: 7.0929, Val AUC: 0.7793
Epoch: 020, Loss: 0.6717, Val AUC: 0.7706
Epoch: 030, Loss: 0.4964, Val AUC: 0.8400
Epoch: 040, Loss: 0.4367, Val AUC: 0.8759
Epoch: 050, Loss: 0.3903, Val AUC: 0.8954
Epoch: 060, Loss: 0.3556, Val AUC: 0.9092
Epoch: 070, Loss: 0.3303, Val AUC: 0.9189
Epoch: 080, Loss: 0.3125, Val AUC: 0.9252
Epoch: 090, Loss: 0.2995, Val AUC: 0.9286
Epoch: 100, Loss: 0.2893, Val AUC: 0.9311

--- Training Complete ---
Final Test AUC: 0.9276


In [34]:
title_mapping = pd.Series(df_movies['title'].values, index=df_movies['id']).to_dict()

def get_movie_recommendations(original_user_id, num_recs=5):
    print(f"--- Fetching Recommendations for User {original_user_id} ---")

    # Put the model in evaluation mode (turns off dropout, gradients, etc.)
    model.eval()

    with torch.no_grad():
        # Get the final, fully-trained embeddings for all nodes in the graph
        trained_x_dict = {
            'movie': model.node_embeddings['movie'].weight,
            'person': model.node_embeddings['person'].weight,
            'genre': model.node_embeddings['genre'].weight,
            'user': model.node_embeddings['user'].weight
        }
        z_dict = model.encoder(trained_x_dict, data.edge_index_dict)

        # Find the internal PyG ID for this specific user
        if original_user_id not in user_mapping:
            return "User not found in the Knowledge Graph."
        user_node_index = user_mapping[original_user_id]

        # Isolate the specific User's embedding [Shape: 64]
        user_emb = z_dict['user'][user_node_index]

        # Isolate ALL Movie embeddings [Shape: num_movies, 64]
        all_movie_embs = z_dict['movie']

        scores = torch.matmul(all_movie_embs, user_emb)

        # Grab all known 'liked' edges from the original full graph
        liked_edges = data['user', 'liked', 'movie'].edge_index

        # Create a boolean mask to find exactly where this user appears as the source
        user_mask = (liked_edges[0] == user_node_index)

        # Extract the PyG movie indices they have already interacted with
        watched_movie_indices = liked_edges[1][user_mask]

        # Force the raw score of watched movies to negative infinity
        scores[watched_movie_indices] = -float('inf')

        # Apply Sigmoid to turn raw scores into probabilities (0.0 to 1.0)
        probabilities = torch.sigmoid(scores)

        # Sort and grab the Top N highest scores
        top_probs, top_indices = torch.topk(probabilities, k=num_recs)

        # Translate PyG internal IDs back to real TMDB Movie IDs
        reverse_movie_mapping = {v: k for k, v in movie_mapping.items()}

        recommendations = []
        for i in range(num_recs):
            pyg_movie_id = top_indices[i].item()
            tmdb_id = reverse_movie_mapping[pyg_movie_id]
            match_score = top_probs[i].item() * 100
            recommendations.append((tmdb_id, match_score))

        return recommendations

In [35]:
# check
def eyeball_test(user_id, num_recs=5):
    print(f"========================================")
    print(f"  PROFILE ANALYSIS FOR USER: {user_id}")
    print(f"========================================\n")

    # WHAT THEY ACTUALLY WATCHED (The Ground Truth)
    # Filter the original dataframe for this user's highly rated movies (e.g., 4.0 or higher)
    user_history = df_user_actions[(df_user_actions['userId'] == user_id) & (df_user_actions['rating'] >= 4.0)]

    print("🎬 WHAT THEY LIKED:")
    for _, row in user_history.head(10).iterrows(): # Just show top 10 for brevity
        tmdb_id = row['tmdbId']
        title = title_mapping.get(tmdb_id, "Unknown")
        if(title != "Unknown"):
            print(f"  - {title} (Rated: {row['rating']})")

    print("\n----------------------------------------\n")

    # WHAT THE MODEL RECOMMENDS (The Prediction)
    print("🤖 MY MODEL RECOMMENDS:")
    recommendations = get_movie_recommendations(user_id, num_recs)

    for rank, (tmdb_id, score) in enumerate(recommendations, 1):
        title = title_mapping.get(tmdb_id, "Unknown")
        if(title != "Unknown"):
          print(f"  {rank}. {title} (Match: {score:.1f}%)")

# Test it on a few different users!
USER_TO_TEST = 1
eyeball_test(USER_TO_TEST)

  PROFILE ANALYSIS FOR USER: 1

🎬 WHAT THEY LIKED:
  - Toy Story (Rated: 4.0)
  - Se7en (Rated: 5.0)
  - The Usual Suspects (Rated: 5.0)
  - Bottle Rocket (Rated: 5.0)
  - Braveheart (Rated: 4.0)
  - Rob Roy (Rated: 5.0)
  - Desperado (Rated: 5.0)

----------------------------------------

🤖 MY MODEL RECOMMENDS:
--- Fetching Recommendations for User 1 ---
  1. Get on Up (Match: 99.6%)
  2. Mortdecai (Match: 99.6%)
  3. Spirited Away (Match: 99.3%)
  4. Terminator 2: Judgment Day (Match: 98.7%)
  5. True Lies (Match: 98.5%)


In [36]:
def calculate_hitrate_at_k(model, full_data, test_data, k=10):
    print(f"--- Calculating HitRate@{k} ---")

    model.eval()

    with torch.no_grad():
        # Generate the freshest embeddings for the entire graph
        trained_x_dict = {
            'movie': model.node_embeddings['movie'].weight,
            'person': model.node_embeddings['person'].weight,
            'genre': model.node_embeddings['genre'].weight,
            'user': model.node_embeddings['user'].weight
        }
        z_dict = model.encoder(trained_x_dict, full_data.edge_index_dict)
        user_embs = z_dict['user']
        movie_embs = z_dict['movie']

        # Extract the "Answer Key" from the test set
        # The test_data contains both real edges (1) and fake negative edges (0).
        # We only care about the real movies the user actually watched.
        test_edge_index = test_data['user', 'liked', 'movie'].edge_label_index
        test_labels = test_data['user', 'liked', 'movie'].edge_label

        # Isolate only the positive edges (where the label is exactly 1.0)
        positive_test_edges = test_edge_index[:, test_labels == 1.0]

        # Group the ground truth test movies by User ID
        # Dictionary format: { user_id: [movie_id_1, movie_id_2] }
        user_to_true_movies = {}
        for i in range(positive_test_edges.size(1)):
            user_id = positive_test_edges[0, i].item()
            movie_id = positive_test_edges[1, i].item()

            if user_id not in user_to_true_movies:
                user_to_true_movies[user_id] = []
            user_to_true_movies[user_id].append(movie_id)

        # Run the Evaluation Loop
        hits = 0
        total_test_users = len(user_to_true_movies)

        for user_id, true_movies in user_to_true_movies.items():
            # Grab this user's 64-D embedding
            u_emb = user_embs[user_id]

            # Score this user against EVERY movie in the database simultaneously
            scores = torch.matmul(movie_embs, u_emb)

            # Get the indices of the Top K highest scoring movies
            # We don't need the actual probabilities here, just the movie IDs
            _, top_k_indices = torch.topk(scores, k=k)
            top_k_movies = top_k_indices.tolist()

            # The Hit Check
            # If ANY of the user's true hidden movies appear in the Top K list, it's a hit!
            if any(m in top_k_movies for m in true_movies):
                hits += 1

        # Calculate the final percentage
        hit_rate = hits / total_test_users

        print(f"Evaluated {total_test_users} users with hidden test data.")
        print(f"Total Hits: {hits}")
        print(f"HitRate@{k}: {hit_rate:.4f} ({(hit_rate * 100):.1f}%)")

        return hit_rate

# HOW TO USE IT (After Training is Complete)
# We usually check Top 10 or Top 20 to simulate the first page of a UI
final_hitrate = calculate_hitrate_at_k(model, data, test_data, k=10)

--- Calculating HitRate@10 ---
Evaluated 608 users with hidden test data.
Total Hits: 229
HitRate@10: 0.3766 (37.7%)
